In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

from armored.models import *
from armored.preprocessing import *

from sklearn.model_selection import KFold

import itertools

from tqdm import tqdm

/home/jcthompson5@ad.wisc.edu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# data with initial and end point measurements
df_exp0 = pd.read_csv("data/exp0/exp0_metabolites.csv")
df_exp1 = pd.read_csv("data/exp1/exp1_metabolites.csv")
df_exp2 = pd.read_csv("data/exp2/exp2_metabolites.csv")
df_exp3 = pd.read_csv("data/exp3/exp3_metabolites.csv")

In [3]:
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

array(['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs',
       'CHabs', 'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs',
       'RIabs', 'pH', 'Lactate', 'Butyrate', 'Acetate', 'AcGum', 'ArGal',
       'Inulin', 'Pectin', 'Starch', 'Xylan'], dtype='<U8')

In [4]:
# separate monoculture data (monoculture always trained on, never used as held-out data)
df_mono = []
df_comm = []

for exp, df_exp in df_exp0.groupby("Experiments"):
    
    # check if monoculture or not
    if sum(df_exp.iloc[df_exp.Time.values==0][species].values[0] > 0) > 1:
        df_comm.append(df_exp)
    else:
        df_mono.append(df_exp)
        
# concatenate dfs
df_mono = pd.concat(df_mono)
df_exp0 = pd.concat(df_comm)

In [5]:
# df for training 
df = pd.concat((df_exp0, df_exp1, df_exp2, df_exp3))

In [6]:
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

n_splits = 20
kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
k_fold_df = pd.DataFrame()
for i, (train_index, test_index) in enumerate(kf.split(unique_treatments)):

    # split into train and test
    train_index = np.in1d(all_treatments, unique_treatments[train_index])
    test_index  = np.in1d(all_treatments, unique_treatments[test_index])
    train_df = df.iloc[train_index].copy()
    test_df  = df.iloc[test_index].copy() 
    
    # add monoculture data
    train_df = pd.concat((df_mono, train_df))
    
    # scale data 
    scaler = MinMaxScaler(observed, system_variables)
    scaler.fit(train_df)
    train_df_scaled = scaler.transform(train_df.copy())
    test_df_scaled = scaler.transform(test_df.copy())
    
    # format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
    train_data = format_data(train_df, species, metabolites, controls, observed=observed)
    train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
    test_data = format_data(test_df, species, metabolites, controls, observed=observed)
    test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

    # instantiate model
    lr = LR(n_species=len(species), 
            n_metabolites=len(metabolites), 
            n_controls=len(controls))
    
    # fit model
    lr.fit(train_data_scaled, 
           alpha_0=0., alpha_1=1.,
           evd_tol=1e-3)

    # make predictions
    predictions = scaler.inverse_transform(lr.predict(test_data_scaled))

    # save predictions
    pred_df = pd.DataFrame()
    for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

        # save species predictions for each experimental condition
        for i, exp_name in enumerate(exp_names):
            # init dataframe
            pred_df_exp = pd.DataFrame()

            # insert exp name
            pred_df_exp["Experiments"] = [exp_name]*len(T[i])
            pred_df_exp["Time"] = T[i]

            for j, s in enumerate(observed):
                pred_df_exp[s + " true"] = Y[i,:,j]
                pred_df_exp[s + " pred"] = preds[i,:,j]
                pred_df_exp[s + " stdv"] = stdvs[i,:,j]

            # append to test prediction dataframe
            pred_df = pd.concat((pred_df, pred_df_exp))
    k_fold_df = pd.concat((k_fold_df, pred_df))
    k_fold_df.to_csv(f"kfold/LR_20_fold_dtl_3.csv", index=False)

Total measurements: 25536, Number of parameters: 494, Initial regularization: 0.00e+00
Loss: 1391.983, Residuals: -0.01255
Loss: 953.602, Residuals: -0.00577
Loss: 850.588, Residuals: -0.00002
Loss: 830.231, Residuals: -0.00008
Loss: 795.099, Residuals: -0.00015
Loss: 758.937, Residuals: -0.00018
Loss: 756.078, Residuals: 0.00033
Loss: 728.368, Residuals: -0.00029
Loss: 717.050, Residuals: -0.00011
Loss: 700.466, Residuals: -0.00042
Loss: 698.758, Residuals: -0.00039
Loss: 696.608, Residuals: -0.00033
Loss: 696.145, Residuals: -0.00045
Loss: 695.948, Residuals: -0.00035
Loss: 689.022, Residuals: -0.00035
Loss: 682.063, Residuals: -0.00060
Loss: 680.689, Residuals: -0.00040
Loss: 677.902, Residuals: -0.00047
Loss: 676.248, Residuals: -0.00058
Loss: 673.751, Residuals: -0.00058
Loss: 672.806, Residuals: -0.00066
Loss: 671.002, Residuals: -0.00064
Loss: 670.658, Residuals: -0.00060
Loss: 670.042, Residuals: -0.00060
Loss: 668.788, Residuals: -0.00063
Loss: 668.411, Residuals: -0.00065
Los

2024-11-04 12:54:13.207842: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25457, Number of parameters: 494, Initial regularization: 0.00e+00
Loss: 1389.047, Residuals: -0.01205
Loss: 956.921, Residuals: -0.00448
Loss: 832.129, Residuals: -0.00207
Loss: 819.866, Residuals: 0.00059
Loss: 797.006, Residuals: 0.00008
Loss: 761.870, Residuals: -0.00067
Loss: 759.385, Residuals: -0.00003
Loss: 736.590, Residuals: -0.00045
Loss: 706.449, Residuals: -0.00083
Loss: 703.208, Residuals: -0.00005
Loss: 700.028, Residuals: -0.00014
Loss: 699.069, Residuals: -0.00038
Loss: 691.525, Residuals: -0.00033
Loss: 690.522, Residuals: -0.00015
Loss: 688.601, Residuals: -0.00019
Loss: 685.432, Residuals: -0.00023
Loss: 683.668, Residuals: -0.00034
Loss: 683.638, Residuals: -0.00047
Loss: 678.989, Residuals: -0.00042
Loss: 673.108, Residuals: -0.00043
Loss: 672.018, Residuals: -0.00025
Loss: 671.265, Residuals: -0.00037
Loss: 669.998, Residuals: -0.00040
Loss: 668.936, Residuals: -0.00060
Loss: 668.800, Residuals: -0.00053
Loss: 667.634, Residuals: -0.00052
Loss

2024-11-04 12:54:48.656280: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25581, Number of parameters: 494, Initial regularization: 0.00e+00
Loss: 1394.284, Residuals: -0.01166
Loss: 963.568, Residuals: -0.00418
Loss: 920.624, Residuals: -0.00084
Loss: 847.587, Residuals: -0.00099
Loss: 779.154, Residuals: -0.00079
Loss: 769.606, Residuals: -0.00011
Loss: 752.428, Residuals: -0.00017
Loss: 707.333, Residuals: -0.00089
Loss: 704.369, Residuals: -0.00092
Loss: 699.966, Residuals: -0.00047
Loss: 699.091, Residuals: -0.00052
Loss: 697.460, Residuals: -0.00057
Loss: 696.799, Residuals: -0.00044
Loss: 690.822, Residuals: -0.00050
Loss: 681.740, Residuals: -0.00044
Loss: 680.137, Residuals: -0.00042
Updating precision...
Evidence 9333.381
Loss: 8570.444, Residuals: -0.00073
Loss: 8481.300, Residuals: -0.00093
Loss: 8463.596, Residuals: -0.00075
Loss: 8435.854, Residuals: -0.00081
Loss: 8421.210, Residuals: -0.00086
Loss: 8408.939, Residuals: -0.00083
Loss: 8405.762, Residuals: -0.00078
Loss: 8398.093, Residuals: -0.00089
Loss: 8397.640, Residual

...

2024-11-04 13:04:33.235809: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25678, Number of parameters: 494, Initial regularization: 0.00e+00
Loss: 1402.029, Residuals: -0.01187
Loss: 965.484, Residuals: -0.00418
Loss: 922.925, Residuals: -0.00089
Loss: 850.475, Residuals: -0.00095
Loss: 782.527, Residuals: -0.00093
Loss: 773.760, Residuals: 0.00001
Loss: 757.553, Residuals: -0.00018
Loss: 726.434, Residuals: -0.00108
Loss: 722.831, Residuals: 0.00039
Loss: 716.041, Residuals: -0.00003
Loss: 704.567, Residuals: -0.00051
Loss: 703.013, Residuals: -0.00030
Loss: 701.761, Residuals: -0.00051
Loss: 701.519, Residuals: -0.00039
Loss: 693.347, Residuals: -0.00044
Loss: 685.079, Residuals: -0.00045
Loss: 683.269, Residuals: -0.00045
Loss: 682.974, Residuals: -0.00050
Loss: 680.378, Residuals: -0.00051
Loss: 676.231, Residuals: -0.00058
Loss: 675.809, Residuals: -0.00054
Loss: 675.203, Residuals: -0.00066
Loss: 675.152, Residuals: -0.00065
Loss: 674.725, Residuals: -0.00063
Loss: 674.590, Residuals: -0.00066
Updating precision...
Evidence 9420.976

2024-11-04 13:05:08.452976: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25476, Number of parameters: 494, Initial regularization: 0.00e+00
Loss: 1384.436, Residuals: -0.01192
Loss: 953.458, Residuals: -0.00408
Loss: 830.540, Residuals: -0.00193
Loss: 818.250, Residuals: 0.00050
Loss: 795.249, Residuals: 0.00004
Loss: 760.324, Residuals: -0.00057
Loss: 758.007, Residuals: 0.00006
Loss: 736.232, Residuals: -0.00031
Loss: 704.758, Residuals: -0.00088
Loss: 700.170, Residuals: -0.00005
Loss: 697.569, Residuals: -0.00050
Loss: 696.613, Residuals: -0.00033
Loss: 689.326, Residuals: -0.00041
Loss: 687.982, Residuals: -0.00042
Loss: 687.687, Residuals: -0.00024
Loss: 684.892, Residuals: -0.00029
Loss: 679.714, Residuals: -0.00037
Loss: 673.100, Residuals: -0.00035
Loss: 671.244, Residuals: -0.00063
Loss: 668.595, Residuals: -0.00046
Loss: 668.101, Residuals: -0.00040
Loss: 667.278, Residuals: -0.00058
Loss: 666.019, Residuals: -0.00053
Loss: 665.881, Residuals: -0.00059
Loss: 665.728, Residuals: -0.00053
Loss: 665.721, Residuals: -0.00055
Updat

2024-11-04 13:05:46.909322: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
